Building a professional MLOps pipeline requires moving beyond simple scripts to an **autonomous system** that monitors its own health. By combining **Evidently** (for drift detection), **GitHub** (for versioned code), and **Prefect** (for orchestration), you create a "closed-loop" system that only retrains when necessary.

# Full Pipeline Orchestration: Drift Detection & Automated Retraining

This guide demonstrates how to schedule a 3-week check that evaluates model health and triggers a retraining cycle only if performance degrades or data drifts.

---

### Step 1: Secure Authentication & GitHub Sync

Before deploying, link your local environment to the cloud services that will manage your secrets and code.

1. **Log in to Prefect Cloud**: Authenticate your CLI.
```bash
uvx prefect-cloud login

```


2. **Connect GitHub**: Authorize Prefect to pull your code from your repository.
```bash
uvx prefect-cloud github setup

```



---

### Step 2: Configure Headless DagsHub Auth

In your `src/model/train.py`, ensure the retraining task can log to MLflow without a browser prompt. Using `dagshub.auth.add_app_token` is essential for remote execution.

```python
import dagshub
import os

def run_train():
    # Use the token injected by Prefect environment variables
    token = os.environ.get("DAGSHUB_TOKEN")
    if token:
        dagshub.auth.add_app_token(token)
    
    # Initialize MLflow tracking on DagsHub
    dagshub.init(repo_name="NGForexCast", repo_owner="Chiebukar", mlflow=True)
    # ... training logic follows

```

---

### Step 3: Deployment with Environment Secrets

Deploy your flow to Prefect Cloud. We use the `--with-requirements` flag to ensure **Evidently**, **LightGBM**, and **Psycopg2** are installed on the remote worker.

**Command:**

```bash
uvx prefect-cloud deploy src/orchestration/flows.py:autonomous_forex_ai \
 --from Chiebukar/NGForexCast \
 --name ngn_forex_autonomous \
 --with-requirements requirements.txt \
 --env DAGSHUB_TOKEN="your_token_here" \
 --env EXCHANGE_RATE_API="your_api_key" \
 --env SUPABASE_DB_URL="your_db_url"

```

---

### Step 4: Schedule the 3-Week "Health Check"

Set an interval schedule to run every 21 days (1,814,400 seconds). This triggers the `autonomous_forex_ai` flow, which first runs the **Evidently** drift check.

**Command:**

```bash
uvx prefect-cloud schedule autonomous_forex_ai/ngn_forex_autonomous --interval 1814400

```

> **Pro Tip**: To schedule different parameters (e.g., a "Deep Retrain" every 6 months vs a "Quick Check" every 3 weeks), you can use [Prefect's per-schedule parameter defaults](https://docs.prefect.io/v3/get-started/github-quickstart#set-per-schedule-parameter-defaults) to reuse a single deployment for multiple purposes.

---

### Step 5: Understanding the "Performance Guard" Logic

Your orchestration uses a nested flow structure to manage state. The `monitor_flow` acts as a gatekeeper:

1. **`run_monitor`**: Executes Evidently drift detection. It compares `reference_data.csv` (training distribution) against `live_df` (recent production data).
2. **Conditional Logic**: If `should_retrain()` returns `True` (due to concept, prediction, or population drift), the `retrain_flow` is triggered.
3. **Retraining**: The new model is trained and logged to DagsHub via MLflow.

---

### Maintenance Commands Reference

| Action | Command |
| --- | --- |
| **Manual Trigger** | `uvx prefect-cloud run autonomous_forex_ai/ngn_forex_autonomous` |
| **Check Logs** | Visit the **Flow Runs** tab in the Prefect Cloud UI. |
| **Delete Deployment** | `uvx prefect-cloud delete autonomous_forex_ai/ngn_forex_autonomous` |

### Why this is Robust

* **Cost Efficient**: Retraining only happens when **Evidently** detects a shift, saving compute resources.
* **Portable**: By using `requirements.txt` and GitHub as source storage, you can switch workers (from GitHub Actions to local servers) without changing your code.
* **Traceable**: Every retraining event is logged as a "Version" in Prefect and a "Run" in DagsHub MLflow.

[Prefect 3.0: Deploying Flows from GitHub](https://www.youtube.com/watch?v=D5DhwVNHWeU)

[Prefect-github starter pack](https://docs.prefect.io/v3/get-started/github-quickstart#set-per-schedule-parameter-defaults)

These resources demonstrates how to connect GitHub repositories to Prefect 3.0 and schedule deployments, which is the exact mechanism used to run your drift checks and retraining.